# ThreadCraft — Measurement Predictor · Step 2: Training

**CPU only.** Set the Kaggle accelerator to **None (CPU)**, Internet **On**, `HF_TOKEN` secret
attached, and run via **Save & Run All (Commit)**.

Trains one gradient-boosted regressor per body measurement, producing two product capabilities:

| Capability | What ThreadCraft does with it |
|---|---|
| **Predict** | Pre-fill wizard Step 4 with sensible editable defaults instead of blank boxes |
| **Validate** | Flag a measurement that contradicts the others before it reaches the tailor |

## The masking design

A real customer supplies an **arbitrary subset** of measurements — maybe just height and weight,
maybe chest and waist too. Training a model on complete rows and then serving it mostly-missing
input is a train/serve mismatch: the model leans on features that won't be there.

So each regressor is trained on **randomly masked copies** of the training data, where random
subsets of the other measurements are blanked out. One model per target then handles any subset,
and gets progressively more accurate as the customer supplies more.

This was measured against the obvious alternative (train on base inputs only) before being
adopted — masking won on **every single target**, even when evaluated with no extra inputs at
all. Section 5 reproduces that comparison.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
HF_USERNAME = "your-hf-username"  # <-- CHANGE THIS (must match 01_data_cleaning.ipynb)

DATASET_REPO_ID = f"{HF_USERNAME}/threadcraft-measurements-cleaned"
MODEL_REPO_ID = f"{HF_USERNAME}/threadcraft-measurement-predictor"

# Always-available inputs: things any customer can give without a tape measure.
BASE_FEATURES = ["height", "weight", "bmi", "age", "sex"]

N_MASKED_COPIES = 4      # augmentation multiplier
MAX_ITER = 300
LEARNING_RATE = 0.08
VALIDATION_PERCENTILE = 99  # residual percentile used as the "implausible" threshold

RANDOM_SEED = 42
PUSH_TO_HUB = True
OUTPUT_DIR = "/kaggle/working/measurement-predictor"

In [ ]:
!pip install -q -U datasets huggingface_hub scikit-learn pandas joblib tabulate

In [ ]:
import os

from huggingface_hub import login

os.makedirs(OUTPUT_DIR, exist_ok=True)

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print(f"Not on Kaggle or secret missing ({e}). Falling back to the HF_TOKEN env var.")
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
elif PUSH_TO_HUB:
    raise RuntimeError("No HF_TOKEN available but PUSH_TO_HUB is True.")

## 1. Load

In [ ]:
import pandas as pd
from datasets import load_dataset

ds = load_dataset(DATASET_REPO_ID)
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

TARGETS = [c for c in train_df.columns if c not in BASE_FEATURES]
print(f"train {train_df.shape}  val {val_df.shape}  test {test_df.shape}")
print(f"\nBase features ({len(BASE_FEATURES)}): {BASE_FEATURES}")
print(f"Targets ({len(TARGETS)}): {TARGETS}")
train_df.head()

## 2. Masking augmentation

For target `T`, the feature set is `BASE_FEATURES + every other measurement`. Augmented copies
blank out random subsets of those other measurements, at a masking rate itself drawn randomly
per copy — so the model sees everything from "nothing but height and weight" through to "almost
everything supplied".

`HistGradientBoostingRegressor` handles `NaN` natively, learning a default split direction for
missing values, so no imputation is involved anywhere.

In [ ]:
import numpy as np

rng = np.random.default_rng(RANDOM_SEED)


def make_masked_training_set(frame, maskable_cols, n_copies=N_MASKED_COPIES):
    """Original rows plus n_copies where random subsets of maskable_cols are NaN."""
    parts = [frame]
    for _ in range(n_copies):
        copy = frame.copy()
        rate = rng.uniform(0.2, 0.9)  # vary how much is hidden per copy
        for col in maskable_cols:
            copy.loc[rng.random(len(copy)) < rate, col] = np.nan
        parts.append(copy)
    return pd.concat(parts, ignore_index=True)


example_others = [t for t in TARGETS if t != TARGETS[0]]
demo = make_masked_training_set(train_df, example_others)
print(f"train rows {len(train_df):,} -> augmented {len(demo):,}")
print(f"missing rate in augmented set: {demo[example_others].isna().mean().mean():.1%}")

## 3. Train one regressor per measurement

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

models = {}
feature_sets = {}

for target in TARGETS:
    others = [t for t in TARGETS if t != target]
    features = BASE_FEATURES + others
    feature_sets[target] = features

    augmented = make_masked_training_set(train_df, others)
    model = HistGradientBoostingRegressor(
        max_iter=MAX_ITER, learning_rate=LEARNING_RATE, random_state=RANDOM_SEED
    )
    model.fit(augmented[features], augmented[target])
    models[target] = model
    print(f"  trained {target:14s} ({len(features)} features, {len(augmented):,} rows)")

print(f"\n{len(models)} regressors trained.")

## 4. Evaluate across realistic input scenarios

A single accuracy number would be meaningless here, because performance depends entirely on how
much the customer supplied. These are the three scenarios the wizard actually produces.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

SCENARIOS = {
    "A: height+weight only": [],
    "B: + chest & waist": ["chest", "waist"],
    "C: + chest, waist, hip, shoulder": ["chest", "waist", "hip", "shoulder"],
}


def predict_target(target, frame, known):
    """Predict `target` given only the measurements listed in `known`."""
    features = feature_sets[target]
    X = frame.reindex(columns=features).copy()
    for col in features:
        if col not in BASE_FEATURES and col not in known:
            X[col] = np.nan
    return models[target].predict(X)


rows = []
for scenario, known in SCENARIOS.items():
    for target in TARGETS:
        if target in known:
            continue  # the customer gave it; nothing to predict
        pred = predict_target(target, test_df, known)
        actual = test_df[target].to_numpy()
        rows.append(
            {
                "scenario": scenario,
                "field": target,
                "r2": r2_score(actual, pred),
                "mae_cm": mean_absolute_error(actual, pred),
            }
        )

results = pd.DataFrame(rows)
pivot_r2 = results.pivot(index="field", columns="scenario", values="r2").round(3)
pivot_mae = results.pivot(index="field", columns="scenario", values="mae_cm").round(2)

print("R-squared by scenario:")
print(pivot_r2.to_string())
print("\nMean absolute error (cm) by scenario:")
print(pivot_mae.to_string())

summary = results.groupby("scenario").agg(mean_r2=("r2", "mean"), mean_mae_cm=("mae_cm", "mean")).round(3)
print("\nOverall:")
print(summary.to_string())

results.to_csv(f"{OUTPUT_DIR}/scenario_results.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
pivot_r2.plot(kind="barh", ax=axes[0], color=["#E8D5C0", "#C4A882", "#8B6B4A"])
axes[0].set_title("Prediction accuracy (R²) by how much the customer supplied")
axes[0].set_xlabel("R²")
axes[0].legend(fontsize=8)

pivot_mae.plot(kind="barh", ax=axes[1], color=["#E8D5C0", "#C4A882", "#8B6B4A"])
axes[1].set_title("Mean absolute error (cm)")
axes[1].set_xlabel("cm")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/accuracy_by_scenario.png", dpi=140)
plt.show()

## 5. Does masking actually beat the simpler alternative?

The obvious alternative is to train on the base inputs only. This compares the two at the hardest
scenario (nothing but height/weight/bmi/age/sex supplied), which is where the base-only model
should be at its most competitive — it was trained for exactly that case.

In [ ]:
comparison_rows = []
for target in TARGETS:
    base_only = HistGradientBoostingRegressor(
        max_iter=MAX_ITER, learning_rate=LEARNING_RATE, random_state=RANDOM_SEED
    ).fit(train_df[BASE_FEATURES], train_df[target])
    r2_base = r2_score(test_df[target], base_only.predict(test_df[BASE_FEATURES]))
    r2_mask = r2_score(test_df[target], predict_target(target, test_df, []))
    comparison_rows.append(
        {"field": target, "base_only_r2": r2_base, "masked_r2": r2_mask, "delta": r2_mask - r2_base}
    )

design_comparison = pd.DataFrame(comparison_rows).set_index("field").round(4)
print(design_comparison.to_string())
wins = (design_comparison["delta"] > 0).sum()
print(f"\nMasking wins on {wins}/{len(design_comparison)} targets "
      f"(mean delta {design_comparison['delta'].mean():+.4f} R²)")
design_comparison.to_csv(f"{OUTPUT_DIR}/design_comparison.csv")

## 6. The validator

The same regressors detect implausible entries. For each field, the residual distribution on
**training** data gives a threshold; at inference, a customer's entry whose deviation from the
predicted value exceeds that threshold is flagged for re-checking.

The threshold percentile is a **precision/recall trade-off and is tuned here on evidence**: too
low and the wizard nags people whose bodies are simply unusual; too high and it misses real
typos.

In [ ]:
# Residual thresholds from the training split, at several candidate percentiles.
CANDIDATE_PCTS = [90, 95, 97.5, 99, 99.5]
thresholds_by_pct = {p: {} for p in CANDIDATE_PCTS}

for target in TARGETS:
    residuals = np.abs(train_df[target].to_numpy() - predict_target(target, train_df, []))
    for p in CANDIDATE_PCTS:
        thresholds_by_pct[p][target] = float(np.percentile(residuals, p))


def evaluate_thresholds(pct, corruption):
    """False-positive rate on clean test values, and catch rate when a value is
    corrupted by `corruption` (e.g. 1.20 = a 20% overstatement)."""
    fps, catches = [], []
    for target in TARGETS:
        pred = predict_target(target, test_df, [])
        actual = test_df[target].to_numpy()
        thr = thresholds_by_pct[pct][target]
        fps.append(float((np.abs(actual - pred) > thr).mean()))
        catches.append(float((np.abs(actual * corruption - pred) > thr).mean()))
    return float(np.mean(fps)), float(np.mean(catches))


print(f"{'pct':>6s} {'false-positive':>15s} {'catch 10% err':>14s} {'catch 20% err':>14s} {'catch 2x err':>13s}")
print("-" * 68)
threshold_report = []
for p in CANDIDATE_PCTS:
    fp, c10 = evaluate_thresholds(p, 1.10)
    _, c20 = evaluate_thresholds(p, 1.20)
    _, c2x = evaluate_thresholds(p, 2.00)
    threshold_report.append({"percentile": p, "false_positive": fp, "catch_10pct": c10,
                             "catch_20pct": c20, "catch_2x": c2x})
    print(f"{p:>6} {fp:>15.1%} {c10:>14.1%} {c20:>14.1%} {c2x:>13.1%}")

pd.DataFrame(threshold_report).to_csv(f"{OUTPUT_DIR}/threshold_tuning.csv", index=False)
print(f"\nUsing the {VALIDATION_PERCENTILE}th percentile (set in the config cell).")
THRESHOLDS = thresholds_by_pct[VALIDATION_PERCENTILE]

In [ ]:
print(f"Flagging thresholds at the {VALIDATION_PERCENTILE}th percentile:")
for target in TARGETS:
    print(f"  {target:14s} deviation > {THRESHOLDS[target]:5.1f} cm from predicted -> flag")

## 7. The two API functions

Reference implementations for `/api/measurements/suggest` and `/api/measurements/validate`.

In [ ]:
def suggest_measurements(customer: dict) -> dict:
    """Given whatever the customer has supplied, predict every field they haven't.
    Requires at minimum height, weight and sex."""
    known = [k for k in TARGETS if customer.get(k) is not None]
    row = {f: customer.get(f, np.nan) for f in set(BASE_FEATURES) | set(TARGETS)}
    if row.get("bmi") is None or (isinstance(row.get("bmi"), float) and np.isnan(row["bmi"])):
        if customer.get("height") and customer.get("weight"):
            row["bmi"] = round(customer["weight"] / (customer["height"] / 100) ** 2, 2)
    frame = pd.DataFrame([row])

    out = {}
    for target in TARGETS:
        if target in known:
            continue
        value = float(predict_target(target, frame, known)[0])
        out[target] = {
            "predicted_cm": round(value, 1),
            "typical_error_cm": round(THRESHOLDS[target] / 2.5, 1),  # rough 1-sigma indication
        }
    return out


def validate_measurements(customer: dict) -> list:
    """Flag supplied measurements that contradict the rest of the profile."""
    supplied = [k for k in TARGETS if customer.get(k) is not None]
    row = {f: customer.get(f, np.nan) for f in set(BASE_FEATURES) | set(TARGETS)}
    if customer.get("height") and customer.get("weight"):
        row["bmi"] = round(customer["weight"] / (customer["height"] / 100) ** 2, 2)
    frame = pd.DataFrame([row])

    warnings = []
    for field in supplied:
        # Predict this field from everything EXCEPT itself.
        others = [k for k in supplied if k != field]
        expected = float(predict_target(field, frame, others)[0])
        deviation = abs(customer[field] - expected)
        if deviation > THRESHOLDS[field]:
            warnings.append(
                {
                    "field": field,
                    "entered_cm": customer[field],
                    "expected_cm": round(expected, 1),
                    "deviation_cm": round(deviation, 1),
                    "message": (
                        f"{field} of {customer[field]} cm looks inconsistent with your other "
                        f"measurements (expected around {expected:.0f} cm). Please re-check."
                    ),
                }
            )
    return warnings


print("--- suggest: 170 cm / 68 kg female, has measured chest and waist ---")
example = {"height": 170.0, "weight": 68.0, "age": 30.0, "sex": 0, "chest": 92.0, "waist": 76.0}
for field, info in suggest_measurements(example).items():
    print(f"  {field:14s} {info['predicted_cm']:6.1f} cm  (+/- ~{info['typical_error_cm']} cm)")

In [ ]:
print("--- validate: a realistic profile (should produce no warnings) ---")
good = {"height": 170.0, "weight": 68.0, "age": 30.0, "sex": 0,
        "chest": 92.0, "waist": 76.0, "hip": 98.0, "sleeve": 82.0}
issues = validate_measurements(good)
print(f"  warnings: {len(issues)}")
for w in issues:
    print("   ", w["message"])

print("\n--- validate: same person, but waist mistyped as 176 (dropped digit) ---")
typo = {**good, "waist": 176.0}
issues = validate_measurements(typo)
print(f"  warnings: {len(issues)}")
for w in issues:
    print("   ", w["message"])

print("\n--- validate: sleeve entered in inches (32) instead of cm (~82) ---")
inches = {**good, "sleeve": 32.0}
issues = validate_measurements(inches)
print(f"  warnings: {len(issues)}")
for w in issues:
    print("   ", w["message"])

## 8. Save and push

In [ ]:
import json

import joblib
import sklearn

artefact = {
    "models": models,
    "feature_sets": feature_sets,
    "base_features": BASE_FEATURES,
    "targets": TARGETS,
    "thresholds": THRESHOLDS,
    "threshold_percentile": VALIDATION_PERCENTILE,
    "units": "cm (weight in kg, age in years, sex 1=male 0=female)",
    "sklearn_version": sklearn.__version__,
}
joblib.dump(artefact, f"{OUTPUT_DIR}/measurement_predictor.joblib")

metrics_blob = {
    "scenarios": json.loads(summary.to_json(orient="index")),
    "per_field_r2": json.loads(pivot_r2.to_json(orient="index")),
    "per_field_mae_cm": json.loads(pivot_mae.to_json(orient="index")),
    "design_comparison": json.loads(design_comparison.to_json(orient="index")),
    "threshold_tuning": threshold_report,
    "n_train": len(train_df), "n_val": len(val_df), "n_test": len(test_df),
    "sklearn_version": sklearn.__version__,
}
with open(f"{OUTPUT_DIR}/metrics.json", "w") as f:
    json.dump(metrics_blob, f, indent=2)
print(f"Saved artefacts to {OUTPUT_DIR}")
print(summary.to_string())

In [ ]:
best_scenario = summary["mean_r2"].idxmax()
worst_field = pivot_r2.iloc[:, 0].idxmin()
chosen_fp, chosen_c20 = evaluate_thresholds(VALIDATION_PERCENTILE, 1.20)

model_card = f"""---
license: other
license_name: us-government-public-domain
tags:
  - tabular-regression
  - anthropometry
  - fashion
datasets:
  - {DATASET_REPO_ID}
metrics:
  - r_squared
  - mae
library_name: sklearn
---

# ThreadCraft Measurement Predictor & Validator

Predicts a person's full set of garment measurements from the few they have actually taken, and
flags entered measurements that contradict the rest of their profile.

Built for [ThreadCraft](https://github.com/Samandee-Galagoda/threadcraft), an AI-powered custom
clothing design and ordering platform, as a final-year BSc Software Engineering project.

## Why it exists

The ThreadCraft proposal names self-reported measurement error as an explicit limitation of the
platform. This model addresses it directly: Step 4 of the design wizard pre-fills editable
suggestions instead of blank boxes, and warns when an entry looks like a typo or a unit mix-up.

## Results (held-out test split, n={len(test_df):,})

Accuracy depends on how much the customer supplied, so it is reported per scenario:

{summary.to_markdown()}

Per-field R² (scenario A = height/weight/age/sex only):

{pivot_r2.to_markdown()}

Mean absolute error, cm:

{pivot_mae.to_markdown()}

## Validator performance

At the {VALIDATION_PERCENTILE}th-percentile residual threshold, averaged across fields:
**{chosen_fp:.1%} false-positive rate** on genuine measurements, catching **{chosen_c20:.1%}** of
values corrupted by 20%. Full percentile sweep in `threshold_tuning.csv`.

## Design: masked-input training

A customer supplies an arbitrary subset of measurements, so each regressor is trained on randomly
masked copies of the data rather than complete rows. This avoids a train/serve mismatch and was
measured against the simpler base-features-only alternative — masking won on
**{wins}/{len(design_comparison)}** targets even at the hardest scenario. See
`design_comparison.csv`.

## Limitations — read before deploying

- **The training population is US Army personnel (ANSUR II, 2012).** They are younger, fitter and
  more athletic than a general civilian population, so predictions will be systematically off for
  older, sedentary or higher-BMI customers, and the extremes of the civilian distribution are
  under-represented.
- **Body proportions vary between populations.** ThreadCraft serves a Sri Lankan market; ANSUR II
  is a US sample. Predicted values should be treated as a **starting point a customer edits**,
  never as a substitute for measuring. This is the single most important caveat here.
- Sex is modelled as the binary recorded in ANSUR II. That is a limitation of the source data,
  and the model should not be presented as covering all customers.
- ANSUR II contains **no bust circumference and no knee circumference**, so those ThreadCraft
  fields cannot be predicted by this model. `chest` is the closest available analogue to bust,
  and `calf` the closest lower-leg circumference.
- `shoulder` maps to biacromial breadth and `inseam` to crotch height — close analogues of the
  tailoring measurements, not identical definitions. Expect a small systematic offset against a
  tailor's own tape.

## Usage

```python
import joblib, numpy as np, pandas as pd
from huggingface_hub import hf_hub_download

art = joblib.load(hf_hub_download("{MODEL_REPO_ID}", "measurement_predictor.joblib"))
models, feature_sets = art["models"], art["feature_sets"]

customer = {{"height": 170.0, "weight": 68.0, "age": 30.0, "sex": 0,
            "chest": 92.0, "waist": 76.0}}
customer["bmi"] = customer["weight"] / (customer["height"] / 100) ** 2
known = [k for k in art["targets"] if k in customer]

for target in art["targets"]:
    if target in known:
        continue
    feats = feature_sets[target]
    row = {{f: customer.get(f, np.nan) for f in feats}}
    for f in feats:
        if f not in art["base_features"] and f not in known:
            row[f] = np.nan
    value = models[target].predict(pd.DataFrame([row])[feats])[0]
    print(f"{{target:14s}} {{value:6.1f}} cm")
```

## Training

| | |
|---|---|
| Algorithm | `HistGradientBoostingRegressor` x {len(TARGETS)} (one per measurement) |
| Augmentation | {N_MASKED_COPIES} masked copies per row, mask rate 20–90% |
| Train / Val / Test | {len(train_df):,} / {len(val_df):,} / {len(test_df):,} (stratified on sex) |
| Hardware | Kaggle CPU (no GPU required) |
| scikit-learn | {sklearn.__version__} |

## Source data

ANSUR II — 2012 Anthropometric Survey of US Army Personnel, public release subset
(4,082 male + 1,986 female, 93 measurements), NATICK/TR-15/007, distributed by the Penn State
OPEN Design Lab. **US Government work, cleared for unlimited public release.**
"""

with open(f"{OUTPUT_DIR}/README.md", "w") as f:
    f.write(model_card)
print(model_card[:1600])

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi

    try:
        api = HfApi(token=HF_TOKEN)
        api.create_repo(MODEL_REPO_ID, exist_ok=True, repo_type="model")
        api.upload_folder(folder_path=OUTPUT_DIR, repo_id=MODEL_REPO_ID, repo_type="model")
        print(f"Pushed: https://huggingface.co/{MODEL_REPO_ID}")
    except Exception as e:
        print(f"PUSH FAILED: {e}")
        print(f"Artefacts are safe in {OUTPUT_DIR} — re-run just this cell to retry.")
else:
    print(f"PUSH_TO_HUB is False — artefacts left in {OUTPUT_DIR}.")

In [ ]:
# Round-trip verification: reload what was pushed and predict with it.
if PUSH_TO_HUB:
    try:
        from huggingface_hub import hf_hub_download

        reloaded = joblib.load(hf_hub_download(MODEL_REPO_ID, "measurement_predictor.joblib"))
        assert reloaded["targets"] == TARGETS
        assert set(reloaded["models"]) == set(models)
        probe = reloaded["models"]["sleeve"]
        print(f"Round-trip OK — {len(reloaded['models'])} regressors reloaded, "
              f"sleeve model has {probe.n_iter_} iterations")
    except Exception as e:
        print(f"Round-trip FAILED: {e}")

## Summary for the dissertation

In [ ]:
print("=" * 70)
print("MEASUREMENT PREDICTOR — RESULTS")
print("=" * 70)
print(f"Source          : ANSUR II, {len(train_df) + len(val_df) + len(test_df):,} people")
print(f"Regressors      : {len(TARGETS)} (one per measurement field)")
print(f"Train/Val/Test  : {len(train_df):,} / {len(val_df):,} / {len(test_df):,}")
print("-" * 70)
for scenario in summary.index:
    print(f"{scenario:34s} R2={summary.loc[scenario, 'mean_r2']:.3f}  "
          f"MAE={summary.loc[scenario, 'mean_mae_cm']:.2f} cm")
print("-" * 70)
print(f"Validator @p{VALIDATION_PERCENTILE}: {chosen_fp:.1%} false positives, "
      f"catches {chosen_c20:.1%} of 20% errors")
print(f"Masking beat base-only on {wins}/{len(design_comparison)} targets")
print("=" * 70)
print(f"\nModel: https://huggingface.co/{MODEL_REPO_ID}")
print("Figures: accuracy_by_scenario.png, scenario_results.csv,")
print("         design_comparison.csv, threshold_tuning.csv")